# DATA preparation

In [61]:
# !python.exe -m pip install --upgrade pip

In [62]:
# !pip install  langchain_community
# !python -m spacy download en_core_web_sm
# !pip install nltk spacy 
# !pip install beautifulsoup4
# !pip install Wikipedia-API
# !pip install googlesearch-python sentence_transformers pandas
# !pip install lxml

In [63]:
import os
import json
import numpy as np  
mushroomenvalv1 = "mushroom.en-val.v2.jsonl"
mushroomenvalv1_output = "./mushroom.en-val.v1.output.json"

num_iteraciones = 2  # Adjust this number as needed
# Seed for reproducibility
seed_val = 442

# =========================
# Setup Environment
# =========================

# Create the output directory if it doesn't exist
output_dir = os.path.dirname(mushroomenvalv1_output)
os.makedirs(output_dir, exist_ok=True)

# =========================
# Load Data
# =========================

# Load the JSON data
data_val_all = []
with open(mushroomenvalv1, 'r', encoding='utf-8') as istr:
    for line in istr:
        try:
            data = json.loads(line.strip())
            if isinstance(data, dict) and 'id' in data:
                data_val_all.append(data)
        except json.JSONDecodeError:
            continue  # Skip invalid lines

num_sample = len(data_val_all)
num_iteraciones = 2  # Adjust this number as needed

print(f"Total de muestras en el conjunto: {num_iteraciones}")

# Adjust the number of iterations to not exceed the total samples
# num_iteraciones = min(num_iteraciones, num_sample)
# num_iteraciones


data_val_all= data_val_all[:num_iteraciones]

Total de muestras en el conjunto: 2


# URL data extraction

In [64]:
from googlesearch import search

#https://medium.com/@karust/now-you-can-search-on-google-for-free-solution-with-api-7699954325b1

# https://github.com/Nv7-GitHub/googlesearch
# Define la consulta de búsqueda
# query = "\u00bfCu\u00e1l es el nombre anterior del actual club de football FC Zhenis?",
# response ="El FC Zhenis es un club de fútbol de Kazajistán de la ciudad de Astaná. Fue fundado en 1946 y jugaba en la Super Liga de Kazajistán hasta 2008 cuando el equipo, al declararse en quiebra, fue relegado a divisiones inferiores. Actualmente juega en la Birinszi Liga."

# Ejecuta la búsqueda
def get_urls(query:str):
    urls = []
    for result in search(query, num_results=8, timeout=10):
        print(result)
        urls.append(result)
    return urls


# urls=get_urls(query)

In [65]:
query = []
response = []
ids = []
for model_input in data_val_all:
    query.append(model_input["model_input"])
    response.append(model_input["model_output_text"])
    ids.append(model_input["id"])
print(query)
print(response)
print(ids)


['What did Petra van Staveren win a gold medal for?', 'How many genera does the Erysiphales order contain?']
['Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.', 'The Elysiphale order contains 5 genera.']
['val-en-1', 'val-en-2']


In [66]:
import pandas as pd

def create_dataframe(query, response, ids):
    """
    Create a DataFrame with query, response and URLs
    
    Args:
        query (str): The search query
        response (str): The response text
        urls (list): List of URLs from search results
    """
    # Create dictionary with the data
    data = {
        'query': query,
        'response': response,
        'id': ids
    }
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Optionally: Create separate columns for each URL
    urls = []
    for text in query:
        urls_text=get_urls(text)
        urls.append([urls_text])
    df["urls"] = urls
        
    return df

# Use the function with your existing variables
df = create_dataframe(query, response, ids)  # Note: query[0] since query is a tuple
df
# Display the DataFrame

https://en.wikipedia.org/wiki/Erysiphales
https://www.sciencedirect.com/topics/agricultural-and-biological-sciences/erysiphaceae
https://www.britannica.com/science/Erysiphales
https://www.researchgate.net/publication/225863704_The_current_systematics_and_taxonomy_of_the_powdery_mildews_Erysiphales_An_overview
https://www.jstor.org/stable/26506945
https://www.cambridge.org/core/books/introduction-to-fungi/hymenoascomycetes-erysiphales/7D2CE4EDF804B4AE4385D6CB7501B001
https://www.tandfonline.com/doi/full/10.3852/13-046
https://www.nepjol.info/index.php/BDPR/article/view/56505/42263


,query,response,id,urls
0,What did Petra van Staveren win a gold medal for?,Petra van Stoveren won a silver medal in the 2...,val-en-1,[[]]
1,How many genera does the Erysiphales order con...,The Elysiphale order contains 5 genera.,val-en-2,"[[https://en.wikipedia.org/wiki/Erysiphales, h..."


In [67]:
from googlesearch import search
import time
import random

def retry_failed_urls(df, max_retries=3, sleep_range=(1, 3)):
    """
    Retry searching for URLs for queries that returned no results
    
    Args:
        df (pd.DataFrame): DataFrame containing the queries and URLs
        max_retries (int): Maximum number of retry attempts
        sleep_range (tuple): Range of seconds to sleep between retries (min, max)
    
    Returns:
        pd.DataFrame: Updated DataFrame with new URLs
    """
    
    def get_urls_with_retry(query, attempt=1):
        try:
            urls = []
            for result in search(query, num_results=3, timeout=10):
                urls.append(result)
            return urls
        except Exception as e:
            if attempt < max_retries:
                # Random sleep to avoid rate limiting
                sleep_time = random.uniform(sleep_range[0], sleep_range[1])
                time.sleep(sleep_time)
                return get_urls_with_retry(query, attempt + 1)
            print(f"Failed to get URLs for query after {max_retries} attempts: {query}")
            print(f"Error: {str(e)}")
            return []

    # Create a copy of the DataFrame
    df_updated = df.copy()
    
    # Find rows with empty or failed URL searches
    failed_searches = df_updated[df_updated['urls'].apply(lambda x: x == [[]] or len(x[0]) == 0)]
    
    print(f"Found {len(failed_searches)} queries with missing URLs")
    
    # Retry failed searches
    for idx in failed_searches.index:
        query = df_updated.loc[idx, 'query']
        print(f"Retrying search for query: {query}")
        
        new_urls = get_urls_with_retry(query)
        if new_urls:
            df_updated.at[idx, 'urls'] = [new_urls]
            print(f"Successfully found {len(new_urls)} URLs")
        else:
            print(f"Still no URLs found for query")
            
        # Add a small delay between searches
        time.sleep(random.uniform(sleep_range[0], sleep_range[1]))
    
    # Print summary
    still_failed = df_updated[df_updated['urls'].apply(lambda x: x == [[]] or len(x[0]) == 0)]
    print(f"\nSummary:")
    print(f"Initially failed searches: {len(failed_searches)}")
    print(f"Remaining failed searches: {len(still_failed)}")
    print(f"Successfully retrieved: {len(failed_searches) - len(still_failed)}")
    
    return df_updated

# Usage example:
df = retry_failed_urls(df)
df

Found 1 queries with missing URLs
Retrying search for query: What did Petra van Staveren win a gold medal for?
Successfully found 3 URLs

Summary:
Initially failed searches: 1
Remaining failed searches: 0
Successfully retrieved: 1


,query,response,id,urls
0,What did Petra van Staveren win a gold medal for?,Petra van Stoveren won a silver medal in the 2...,val-en-1,[[https://en.wikipedia.org/wiki/Petra_van_Stav...
1,How many genera does the Erysiphales order con...,The Elysiphale order contains 5 genera.,val-en-2,"[[https://en.wikipedia.org/wiki/Erysiphales, h..."


In [68]:
df["urls"][0]

[['https://en.wikipedia.org/wiki/Petra_van_Staveren',
  'https://olympics.fandom.com/wiki/Petra_van_Staveren',
  'https://www.worldaquatics.com/athletes/1074710/petra-van-staveren']]

In [69]:
from langchain_community.document_loaders import WebBaseLoader
import pandas as pd
import logging
from typing import List


def page_content(infos):
    """
    Extract page content from document objects
    
    Args:
        infos: List of Document objects or a single Document object
    Returns:
        List of extracted text content
    """
    page_content = []
    
    # Handle empty input
    if not infos:
        return page_content
        
    # If infos is a single Document object
    if hasattr(infos, 'page_content'):
        return [infos.page_content]
    
    # If infos is a list of Document objects
    for info in infos:
        try:
            if hasattr(info, 'page_content'):
                page_content.append(info.page_content)
        except Exception as e:
            print(f"Error extracting content: {str(e)}")
            continue
            
    return page_content


def load_content(urls_list: List) -> List:
    """
    Load content from a list of URLs using WebBaseLoader
    
    Args:
        urls_list: List of URLs to process
    Returns:
        List of extracted content
    """
    if not urls_list or urls_list == [[]]:
        return []
    
    # Flatten the nested list if necessary
    if isinstance(urls_list[0], list):
        urls_list = urls_list[0]
    
    docs = []
    for url in urls_list:
        try:
            loader = WebBaseLoader(url)
            doc = loader.load()
            docs.extend(doc)
            # Add small delay to avoid overwhelming servers
            # time.sleep(0.2)
        except Exception as e:
            logging.warning(f"Error loading content from {url}: {str(e)}")
            continue
    
    return docs

def update_dataframe_with_content(df: pd.DataFrame) -> pd.DataFrame:
    """
    Update DataFrame with extracted content from URLs
    
    Args:
        df: Input DataFrame with 'urls' column
    Returns:
        Updated DataFrame with 'info' and 'content' columns
    """
    # Create a copy to avoid SettingWithCopyWarning
    df_updated = df.copy()
    
    # Add info column
    df_updated['scraping_and_procesor'] = df_updated['urls'].apply(load_content)
    return df_updated


In [70]:
# no es necesario ejecutar esta celda porque ya se está extrayendo el contenido despues con  text_extractor
# df = update_dataframe_with_content(df)
# df["content"] = df["info"].apply(page_content)
# df["content"][0]

# Mejorar la logica del retriver para obtener la información más relevante.

In [71]:
from bs4 import BeautifulSoup, Comment
import urllib.request
import pandas as pd
import time

def text_extractor(urls):
    def tag_visible(element):
        if element.parent.name in [
                'style', 'script', 'head', 'title', 'meta', '[document]'
        ]:
            return False
        if isinstance(element, Comment):
            return False
        return True

    def text_from_html(url):
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) ' \
                          'AppleWebKit/537.36 (KHTML, like Gecko) ' \
                          'Chrome/58.0.3029.110 Safari/537.3',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'
        }
        req = urllib.request.Request(url, headers=headers)
        try:
            with urllib.request.urlopen(req, timeout=10) as response:
                body = response.read()
        except urllib.error.HTTPError as e:
            print(f"HTTP Error {e.code} for URL: {url}")
            return ""
        except urllib.error.URLError as e:
            print(f"URL Error: {e.reason} for URL: {url}")
            return ""

        soup = BeautifulSoup(body, 'html.parser')
        texts = soup.findAll(text=True)
        visible_texts = filter(tag_visible, texts)
        return " ".join(t.strip() for t in visible_texts)

    texts = []
    # Assuming urls is a list of lists
    for idx, url in enumerate(urls[0]):
        if url is not None:
            text = text_from_html(url)
            texts.append(text)
            time.sleep(1)
        else:
            texts.append("")
        if idx % 10 == 0:
            print(f"Processed {idx} URLs")
    
    return texts

# Apply the text_extractor function to the 'urls' column

In [72]:
#no es necesario
# df["scraping_and_procesor"] = df["urls"].apply(text_extractor)

# len(df["scraping_and_procesor"][0])

In [73]:
import requests
from bs4 import BeautifulSoup
import json

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'DNT': '1',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1'
}

def google_search(query):
    num_results=10
    params = {
        'q': query,
        'gl': 'us',
        'num': num_results
    }
    
    try:
        # Make the request
        response = requests.get(
            'https://www.google.com/search',
            headers=headers,
            params=params,
            timeout=30
        )
        response.raise_for_status()  # Raise an exception for bad status codes
        
        # Parse the HTML
        soup = BeautifulSoup(response.text, 'lxml')
        
        # Extract search results
        search_results = []
        
        # Find all search result divs
        for result in soup.select('div.g'):
            try:
                # Extract title
                title_element = result.select_one('h3')
                title = title_element.text if title_element else None
                
                # Extract link
                link_element = result.select_one('a')
                link = link_element['href'] if link_element else None
                
                # Extract snippet
                snippet_element = result.select_one('span.hgKElc') or result.select_one('.VwiC3b')
                snippet = snippet_element.text if snippet_element else None
                
                # Only add results that have at least a title or snippet
                if title or snippet:
                    search_results.append({
                        'title': title,
                        'link': link,
                        'snippet': snippet
                    })
                    
            except Exception as e:
                print(f"Error parsing individual result: {str(e)}")
                continue
        
        return search_results
    
    except requests.RequestException as e:
        print(f"Error making request: {str(e)}")
        return []
    except Exception as e:
        print(f"Unexpected error: {str(e)}")
        return []

def print_results(results):
    if not results:
        print("No results found.")
        return
    
    for i, result in enumerate(results, 1):
        print(f"\nResult {i}:")
        print(f"Title: {result['title']}")
        print(f"Link: {result['link']}")
        print(f"Snippet: {result['snippet']}")
        print("-" * 80)


def save_results(results, filename='results1.json'):
    try:
        # Print current working directory and results for debugging
        import os
        print(f"Current working directory: {os.getcwd()}")
        print(f"Attempting to save {len(results)} results")
        
        # Add absolute path to the file
        file_path = os.path.join(os.getcwd(), filename)
        
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        
        # Verify file was created
        if os.path.exists(file_path):
            print(f"File successfully saved at: {file_path}")
        else:
            print("File was not created")
            
    except Exception as e:
        print(f"Error saving results: {str(e)}")

# # Example usage
# if __name__ == "__main__":
#     query = "Who was the father of Carl L\u00f6wenhielm?"
#     results = google_search(query)
    
#     # Debug print before saving
#     print(f"Number of results found: {len(results)}")
#     print_results(results)
#     save_results(results, filename='results.json')

In [74]:
df

,query,response,id,urls
0,What did Petra van Staveren win a gold medal for?,Petra van Stoveren won a silver medal in the 2...,val-en-1,[[https://en.wikipedia.org/wiki/Petra_van_Stav...
1,How many genera does the Erysiphales order con...,The Elysiphale order contains 5 genera.,val-en-2,"[[https://en.wikipedia.org/wiki/Erysiphales, h..."


In [75]:
df["snippes_google"] = df["query"].apply(google_search)

In [76]:
df["snippes_google"]

0    [{'title': 'Petra van Staveren - Wikipedia', '...
1    [{'title': 'Erysiphales - an overview | Scienc...
Name: snippes_google, dtype: object

In [77]:
def fix_dict_to_list(list_of_dicts):
    lista=[]
    for dicts in list_of_dicts:
        if dicts["title"]:
            lista.append(dicts["title"])
        if dicts["snippet"]:
            lista.append(dicts["snippet"])
    return lista

In [78]:
df["scraping_and_procesor"] = df["snippes_google"].apply(fix_dict_to_list)
df["scraping_and_procesor"][0]

['Petra van Staveren - Wikipedia',
 'Petronella ("Petra") Grietje van Staveren (born 2 June 1966) is a former breaststroke swimmer from the Netherlands who won the gold medal in the 100 meter breaststroke at the 1984 Summer Olympics in Los Angeles.',
 'Petra van Staveren | Olympics Wiki - Fandom',
 'Van Staveren arrived at her first Olympics in 1984 as the medal outsider for the breaststroke double (100 and 200). In the 100 metres breaststroke, she turned a\xa0...',
 'Petra van Staveren',
 'Gold. 200 metres Breaststroke, Women (Olympic), 10. 4 × 100 metres Medley ... Listed in Olympians Who Won a Medal at the European Aquatics Championships\xa0...',
 'Petra VAN STAVEREN | Results',
 'Personal Best Results ; Women 100 Breaststroke, 01:09.88, Gold, 50m, 18 ; Women 200 Breaststroke, 02:36.14, -, 50m, 17\xa0...',
 'Petra van Staveren Olympic Medals',
 'Petra van Staveren has won 1 Olympic medal in throughout her career. When did Petra van Staveren first compete in the Olympics? She represe

In [79]:
df["scraping_and_procesor"][0]

['Petra van Staveren - Wikipedia',
 'Petronella ("Petra") Grietje van Staveren (born 2 June 1966) is a former breaststroke swimmer from the Netherlands who won the gold medal in the 100 meter breaststroke at the 1984 Summer Olympics in Los Angeles.',
 'Petra van Staveren | Olympics Wiki - Fandom',
 'Van Staveren arrived at her first Olympics in 1984 as the medal outsider for the breaststroke double (100 and 200). In the 100 metres breaststroke, she turned a\xa0...',
 'Petra van Staveren',
 'Gold. 200 metres Breaststroke, Women (Olympic), 10. 4 × 100 metres Medley ... Listed in Olympians Who Won a Medal at the European Aquatics Championships\xa0...',
 'Petra VAN STAVEREN | Results',
 'Personal Best Results ; Women 100 Breaststroke, 01:09.88, Gold, 50m, 18 ; Women 200 Breaststroke, 02:36.14, -, 50m, 17\xa0...',
 'Petra van Staveren Olympic Medals',
 'Petra van Staveren has won 1 Olympic medal in throughout her career. When did Petra van Staveren first compete in the Olympics? She represe

In [80]:
df.to_csv("url_content.csv", index=False)


In [48]:
# import pandas as pd
# import ast  # For safely evaluating strings containing Python expressions

# def parse_list_string(s):
#     try:
#         # Safely evaluate the string representation of the list
#         return ast.literal_eval(s)
#     except (ValueError, SyntaxError):
#         return []

# # Read the CSV file
# df_copy = pd.read_csv("url_content.csv")

# # Parse the string representation of lists in the scraping_and_procesor column
# df_copy["scraping_and_procesor"] = df_copy["scraping_and_procesor"].apply(parse_list_string)


In [82]:
df["query"][0]

'What did Petra van Staveren win a gold medal for?'

In [83]:
df["scraping_and_procesor"][0][0]

'Petra van Staveren - Wikipedia'

In [84]:
from sentence_transformers import SentenceTransformer, util
model_embeddings = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1') # 1billon parameters


def sentence_transformers_feature(text1, text2):
    """
    Calculate normalized cosine similarity between two texts
    
    Args:
        text1 (str): First text
        text2 (str): Second text
        
    Returns:
        float: Normalized similarity score between 0 and 1
              1 = most similar, 0 = most different
    """
    # Encode texts
    embedding_1 = model_embeddings.encode(text1)
    embedding_2 = model_embeddings.encode(text2)
    
    # Calculate cosine similarity
    cosine_similarity = util.pytorch_cos_sim(embedding_1, embedding_2)
    
    # Convert to scalar value
    similarity_score = cosine_similarity[0].item()
    
    # Normalize to [0, 1] range
    normalized_score = (similarity_score + 1) / 2
    
    return normalized_score


In [85]:
model_embeddings1 = SentenceTransformer('all-distilroberta-v1') # 1billon parameters
# multi-qa-mpnet-base-cos-v1
def sentence_transformers_featurev2(text1, text2):
    """model to evaluate the probability of hallucination"""
    embedding_1= model_embeddings1.encode(text1)
    embedding_2 = model_embeddings1.encode(text2)
    cosine_similarity = util.pytorch_cos_sim(embedding_1, embedding_2)
    tensor1 = (cosine_similarity[0]).item()
    normalized_score = (tensor1 + 1) / 2
    tensor1 = 1 - normalized_score
    return tensor1

word1= "See how many days are left in 2025. There are 363 days remaining in the year. That's equivalent to 51 weeks and 6 days left until the end of 2025."
word2 = "If today is 14th October, and it is not a leap year, how many days remain until the end of the year?"
similarity = sentence_transformers_featurev2(word1,word2)
similarity  

0.24180367588996887

In [86]:
def evalaute_text_similarity(row):
    input_question = row['query']
    texts1 = row['scraping_and_procesor']
    result = []
    for text in texts1:
        resultado = sentence_transformers_feature(text,input_question)
        result.append(resultado)
    return result

# df["urls_similarity_default"] = df.apply(evalaute_text_, axis=1)
# df

In [87]:
import requests
def pipeline_feature(text):
    preprocessing_methods = {
    # "syntax_patterns_spacy": "special",
    # "syntax_patterns": "special",
    # "token_frequency": "special",
    # "stemming": "text",
    # "lemmatization": "text",
    "remove_stopwords": "text",
    # "proper_encoding_ascii": "text",
    #"proper_encoding_utf_8": "text",
    "clean_text": "text"
    }
    data = {
        "text_list": 
            [text]
        ,
        "lang": "en",
        "preprocessing_methods": preprocessing_methods
        }
    
    
    response = requests.post(f"http://localhost:8000/feature/pipeline/", json=data, timeout=10)
    return response.json()["text"]
#response1 = requests.post(f"http://localhost:8000/feature/pipeline/", json=data)

def fix_pipeline_list_txt(texts):
    list_text = []
    for text in texts:
        list_text.append(pipeline_feature(text))
    return list_text





In [88]:
#X_pre = response.json()["text"]
df["clean_pipeline_web_v"] = df["scraping_and_procesor"].apply(fix_pipeline_list_txt)


In [89]:
df["clean_pipeline_web_v"][0]

[['petra van staveren wikipedia'],
 ['petronella petra grietje van staveren born june breaststroke swimmer netherlands won gold medal meter breaststroke summer olympics los angeles .'],
 ['petra van staveren olympics wiki fandom'],
 ['van staveren arrived olympics medal outsider breaststroke double . metres breaststroke turned ...'],
 ['petra van staveren'],
 ['gold . metres breaststroke women olympic . metres medley ... listed olympians won medal european aquatics championships ...'],
 ['petra van staveren results'],
 ['personal best results women breaststroke gold m women breaststroke m ...'],
 ['petra van staveren olympic medals'],
 ['petra van staveren won olympic medal career . petra van staveren compete olympics represented netherlands ...'],
 ['los angeles swimming olympic results discipline'],
 ['official swimming results los angeles olympics . list gold silver bronze medallists photos videos medal winning ...'],
 ['petra van staveren xxiii summer olympics'],
 ['feb — dutch swi

In [90]:
def delete_duplicates(data):
    deduplicated_list = list(set(item[0] for item in data))
    return deduplicated_list

df["clean_pipeline"] = df["clean_pipeline_web_v"].apply(delete_duplicates)
df["clean_pipeline"][0]

['final women metre breaststroke event summer olympics held mcdonald olympic swim stadium los angeles ...',
 'petra van staveren biography',
 'petra van staveren',
 'official swimming results los angeles olympics . list gold silver bronze medallists photos videos medal winning ...',
 'van staveren arrived olympics medal outsider breaststroke double . metres breaststroke turned ...',
 'gold . metres breaststroke women olympic . metres medley ... listed olympians won medal european aquatics championships ...',
 'petra van staveren wikipedia',
 'petra van staveren results',
 'petra van staveren olympic medals',
 'petronella petra grietje van staveren born june breaststroke swimmer netherlands won gold medal meter breaststroke summer olympics los angeles .',
 'petra van staveren won olympic medal career . petra van staveren compete olympics represented netherlands ...',
 'personal best results women breaststroke gold m women breaststroke m ...',
 'swimming summer olympics – women ...',
 'p

In [91]:
def split_into_chunks(text):
    # Split the text into words
    try:
        # Initialize variables
        chunks = []
        current_chunk = []
        word_count = 0
        
        # Iterate through words and create chunks
        for word in words:
            current_chunk.append(word)
            word_count += 1
            
            # When we reach 100 words, add the chunk to our results
            if word_count == words_per_chunk:
                chunks.append(' '.join(current_chunk))
                current_chunk = []
                word_count = 0
        
        # Add the remaining words as the last chunk (if any)
        if current_chunk:
            chunks.append(' '.join(current_chunk))
        return chunks
    except Exception as e:
        return ""

        

def list_to_chunks(list_words):
    # print(len(list_words))
    result = []
    for text in list_words:
        
        # print(f"texto {text[0]}")
        chunks = split_into_chunks(text[0])
        result.append(chunks)
        # print(result)
    return result

# # Example usage
# text = "Your long text goes here... (imagine lots of words)"
# chunks = split_into_chunks(text)

# # Print each chunk with its word count
# for i, chunk in enumerate(chunks, 1):
#     print(f"Chunk {i} ({len(chunk.split())} words):")
#     print(chunk)

#     print("-" * 50)


# df["chunks"] =  df["clean_pipeline"].apply(list_to_chunks)
# df["chunks"]


In [92]:
df["clean_pipeline"]

0    [final women metre breaststroke event summer o...
1    [taxonomic manual erysiphales powdery mildews,...
Name: clean_pipeline, dtype: object

In [93]:

from sentence_transformers import SentenceTransformer, util
model_embeddings1 = SentenceTransformer('multi-qa-mpnet-base-dot-v1') # 1billon parameters
# multi-qa-mpnet-base-cos-v1
def query_chuncks_similarity(query, text2):
    """model to evaluate the probability of hallucination"""
    embedding_1= model_embeddings1.encode("Question: " + query)
    embedding_2 = model_embeddings1.encode("Answer: " + text2)
    cosine_similarity = util.pytorch_cos_sim(embedding_1, embedding_2)
    tensor1 = (cosine_similarity[0]).item()
    # normalized_score = (tensor1 + 1) / 2
    # tensor1 = 1 - normalized_score
    result = {"chuck": text2, "score": tensor1}
    # print(result)
    return result


def chunk_ranking(row):
    query = row['query']
    result = []
    for chunk in row['clean_pipeline']:
        similarity = query_chuncks_similarity(query,chunk)
        result.append(similarity)
    sorted_data = sorted(result, key=lambda x: x['score'], reverse=True)
    return sorted_data

df["chunk_ranking"] = df.apply(chunk_ranking, axis=1)



In [149]:
df["chunk_ranking"][0]

[{'chuck': 'petra van staveren olympic medals', 'score': 0.8068445920944214},
 {'chuck': 'petra van staveren won olympic medal career . petra van staveren compete olympics represented netherlands ...',
  'score': 0.7774338126182556},
 {'chuck': 'petra van staveren olympics wiki fandom',
  'score': 0.7682787775993347},
 {'chuck': 'petra van staveren xxiii summer olympics',
  'score': 0.7588487863540649},
 {'chuck': 'feb — dutch swimmer petra van staveren celebrates finishing win gold medal final women metres breaststroke ...',
  'score': 0.7188681364059448},
 {'chuck': 'petra van staveren', 'score': 0.6761782765388489},
 {'chuck': 'competed summer olympic games . van staveren arrived olympics medal outsider breaststroke double ...',
  'score': 0.6741144061088562},
 {'chuck': 'petra van staveren results', 'score': 0.662869930267334},
 {'chuck': '1st place gold medalists anne ottenbrite ... prior competition existing world olympic records follows . ... petra van staveren .',
  'score': 0.

In [94]:
df["chunk_ranking"][1]

[{'chuck': 'date sixteen genera containing species described erysiphales order braun cook .',
  'score': 0.755731999874115},
 {'chuck': 'nov — species comprise erysiphaceae family family order erysiphales . fungi comprise species ...',
  'score': 0.7469596862792969},
 {'chuck': 'erysiphales order fungi', 'score': 0.7445911169052124},
 {'chuck': 'erysiphaceae overview', 'score': 0.7168132066726685},
 {'chuck': 'taxonomic manual erysiphales powdery mildews',
  'score': 0.6688570380210876},
 {'chuck': 'erysiphaceae molecular phylogenetic relationships dna based species genera delimitations based nuclear ribosomal ...',
  'score': 0.6645367741584778},
 {'chuck': 's takamatsu · · cited — erysiphaceae consists tribes basal genera . tribes include tree parasitic herb parasitic species . tree ...',
  'score': 0.6400007009506226},
 {'chuck': 'systematics . order contains family erysiphaceae genera species . imperfect fungi fungi sexual reproduction unknown ...',
  'score': 0.6397525668144226},


# validate transformer

In [170]:
#validate that the chuck ranking is working
# multi-qa-mpnet-base-cos-v1
def query_chuncks_similarity(text1, text2):
    """model to evaluate the probability of hallucination"""
    model_embeddings1 = SentenceTransformer('multi-qa-mpnet-base-dot-v1') # multi-qa-mpnet-base-cos-v1
    embedding_1= model_embeddings1.encode(text1)
    embedding_2 = model_embeddings1.encode(text2)
    cosine_similarity = util.pytorch_cos_sim(embedding_1, embedding_2)
    tensor1 = (cosine_similarity[0]).item()
    # normalized_score = (tensor1 + 1) / 2
    # tensor1 = 1 - normalized_score
    result = {"chuck": text2, "score": tensor1}
    return result

word1= "This answer solve the question: Elon Musk has 12 children with three women"
word2 = "question: how many children does elon musk have?"


word1= "This answer solve the question: van Staveren won the gold medal in the 100 meter breaststroke"
word2 = "question: What did Petra van Staveren win a gold medal for?"

word1= "answer: Elon Musk has 12 children with three women"
word2 = "question: how many children does elon musk have?"
  



word1= "answer: there are 78 days remaining until the end of the year."
word2 = "question: If today is 14th October, and it is not a leap year, how many days remain until the end of the year?"

similarity = query_chuncks_similarity(word1,word2)
similarity

{'chuck': 'question: If today is 14th October, and it is not a leap year, how many days remain until the end of the year?',
 'score': 0.6948051452636719}

# extraer los texto más relevantes y crear los textos que no se le creó la relevancia 

In [113]:
def eliminate_not_significated_chunks(lista):
    """Eliminar los chunks que no tienen relevancia más de 65%, si no return none"""
    result = []
    for dictionary in lista: 
        chuck = dictionary['chuck'] 
        score = dictionary['score'] 
        if score > 0.65:
            result.append(chuck)
    return result


df["chunks_filter"] = df["chunk_ranking"].apply(eliminate_not_significated_chunks)

In [114]:
df["chunks_filter"][0]

['petra van staveren olympic medals',
 'petra van staveren won olympic medal career . petra van staveren compete olympics represented netherlands ...',
 'petra van staveren olympics wiki fandom',
 'petra van staveren xxiii summer olympics',
 'petra van staveren biography',
 'feb — dutch swimmer petra van staveren celebrates finishing win gold medal final women metres breaststroke ...',
 'petra van staveren wikipedia',
 'petra van staveren',
 'petronella petra grietje van staveren born june breaststroke swimmer netherlands won gold medal meter breaststroke summer olympics los angeles .',
 'petra van staveren results']

In [115]:
df.to_csv("chunks_filter.csv", index=False)


# answer generetor by LLS

In [1]:
import pandas as pd
import ast  # For safely evaluating strings containing Python expressions

def parse_list_string(s):
    try:
        # Safely evaluate the string representation of the list
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return []

# # Read the CSV file
df_copy_v2 = pd.read_csv("chunks_filter.csv")

# Parse the string representation of lists in the scraping_and_procesor column
df_copy_v2["chunks_filter"] = df_copy_v2["chunks_filter"].apply(parse_list_string)

In [2]:
df_copy_v2["chunks_filter"][0]

['alberto fouillioux international appearances',
 'alberto fouillioux soccer player profile career statistics',
 'alberto fouillioux player',
 'alberto fouillioux wikipedia',
 'alberto fouillioux biography',
 'stats alberto fouillioux world cup qualification south america world cup qualification · · world cup world cup · · ...',
 'alberto fouillioux international appearances . outstanding chilean player obtained international caps national team .',
 'alberto fouillioux national team transfermarkt']

In [119]:
# # Debemos importar la librería de OpenAI

# # Creamos un cliente que apunta a la dirección local de LM Studio
# client = OpenAI(base_url="http://localhost:5000/v1", api_key="not-needed")

# completion = client.chat.completions.create(
#   model="mistral-7b-instruct-v0.2:2", # Este campo no se utiliza
#   messages=[
#     {"role": "assistant", "content": "Eres un asistente que habla sobre temas con dos palabras"},
#     {"role": "user", "content": "Hablame sobre el movimiento maker y su rol durante la pandemia."}
#   ],
#   temperature=0.7,
#   logprobs=True,
#   top_logprobs = 1
  
# )

# print(completion.choices[0].message)
# print(completion)



KeyboardInterrupt



In [14]:
from openai import OpenAI

def generate_response(prompt, system_message="answer the question"):
    """
    Generate a response using LM Studio's local API
    
    Args:
        prompt (str): The user's input text
        system_message (str): Instructions for the AI assistant
        
    Returns:
        str: The generated response text, or None if there's an error
    """
    try:
        # Initialize client
        client = OpenAI(
            base_url="http://localhost:5000/v1", 
            api_key="not-needed"
        )
        
        # Generate completion
        response = client.chat.completions.create(
            model="mistral-7b-instruct-v0.2:2",  # Model name not used by LM Studio
            messages=[
                {"role": "assistant", "content": system_message},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )
        
        # Return just the response text
        return response.choices[0].message.content
        
    except Exception as e:
        print(f"Error: {str(e)}")
        return None

# Example usage:
"""
response = generate_response(
    prompt="What is machine learning?",
    system_message="You are an AI expert"
)
print(response)
"""

'\nresponse = generate_response(\n    prompt="What is machine learning?",\n    system_message="You are an AI expert"\n)\nprint(response)\n'

In [15]:
def generate_answers(row):
    result = []
    list_of_text = row['chunks_filter']
    if len(list_of_text) < 3:
        response = generate_response(
                prompt=row['query'],
        )
        # Handle None response
        if response is not None:
            if isinstance(response, (list, tuple)):
                result.extend(response)
            else:
                result.append(response)
    else:
        result = list_of_text
    return result

# Apply the function row by row
df_copy_v2["chunks_filter_v2"] = df_copy_v2.apply(generate_answers, axis=1)

In [8]:
# def generate_answers(list_of_text):
#     result = []
#     if len(list_of_text)<3:
#         for text in list_of_text:
#             response = generate_response(
#                 prompt=text,
#             )
#             result.extend(response)
#     else:
#         result = list_of_text
#     return result
        
        

# df_copy_v2["chunks_filter_v2"] = df_copy_v2.apply(generate_answers)

ValueError: Cannot set a DataFrame with multiple columns to the single column chunks_filter_v2

In [16]:
df_copy_v2.to_csv("generated_answers3.csv", index=False)


# analisis de con stopword de valor del contido de las páginas

In [121]:
import pandas as pd
import ast  # For safely evaluating strings containing Python expressions

def parse_list_string(s):
    try:
        # Safely evaluate the string representation of the list
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return []

# # Read the CSV file
df = pd.read_csv("generated_answers3.csv")

# Parse the string representation of lists in the scraping_and_procesor column
df["answers"] = df["chunks_filter_v2"].apply(parse_list_string)

In [122]:
df["answers"] 

0    [petra van staveren olympic medals, petra van ...
1    [date sixteen genera containing species descri...
Name: answers, dtype: object

In [123]:
import pandas as pd
import csv
file = open('Eng_GoogleUnigrams.csv', "r", encoding='utf-8')
N_Gram_Google = list(csv.reader(file, delimiter=";"))
file.close()
N_Gram_Google = pd.DataFrame(N_Gram_Google)
N_Gram_Google

,0,1
0,a.165,3.08
1,a.d.a.a.,5.79
2,a.g.s,3.38
3,a.j.u.,2.91
4,a.k.k.,5.33
...,...,...
6156632,zuriickfiihren,4.38
6156633,zuweit,1.87
6156634,zwelff,2.75
6156635,zwierzę,2.61


# Loading NLP 

In [124]:
def delete_none(texts):
    # If input is None, return empty string
    if texts is None:
        return ""
    
    # Handle nested lists
    if isinstance(texts, list):
        # Recursively process nested lists
        return [delete_none(item) for item in texts]
    
    # Return the text itself if it's not None, otherwise empty string
    return texts if texts is not None else ""

# Apply the function
df["clean"] = df["clean_pipeline_web_v"].apply(delete_none)

In [125]:
df["clean"]

0    [['petra van staveren wikipedia'], ['petronell...
1    [['erysiphales overview sciencedirect topics']...
Name: clean, dtype: object

# Execution

In [128]:
def fix_sentence(lista):
    lista = ", ".join(lista)
    return lista

df["fix_lista"] = df["answers"].apply(fix_sentence)
df["fix_lista"][0]

'petra van staveren olympic medals, petra van staveren won olympic medal career . petra van staveren compete olympics represented netherlands ..., petra van staveren olympics wiki fandom, petra van staveren xxiii summer olympics, petra van staveren biography, feb — dutch swimmer petra van staveren celebrates finishing win gold medal final women metres breaststroke ..., petra van staveren wikipedia, petra van staveren, petronella petra grietje van staveren born june breaststroke swimmer netherlands won gold medal meter breaststroke summer olympics los angeles ., petra van staveren results'

In [135]:
import pandas as pd
from collections import Counter
import re

def create_word_frequency_df(row):
    # Get text from clean_scraping column and flatten the nested list
    
    sentences = row['fix_lista'] # Access first element since it's a nested list
    
    # Join all sentences into a single text
    if isinstance(sentences, list):
        text = ' '.join(sentences)
    else:
        text = sentences
        
    # Convert to lowercase and split into words
    # Remove punctuation and special characters
    words = re.findall(r'\b\w+\b', text.lower())
    
    # Count frequencies
    word_counts = Counter(words)
    
    # Create DataFrame and sort by frequency
    df = pd.DataFrame.from_dict(word_counts, orient='index', columns=['Count'])
    df = df.reset_index().rename(columns={'index': 'Words'})
    df = df.sort_values('Count', ascending=False)
    df = df.reset_index(drop=True)
    
    return df

# Apply the function to create word frequency DataFrames
df["word_frequency"] = df.apply(create_word_frequency_df, axis=1)
df["word_frequency"][0]


,Words,Count
0,petra,11
1,staveren,11
2,van,11
3,olympics,4
4,breaststroke,3
5,medal,3
6,swimmer,2
7,summer,2
8,gold,2
9,netherlands,2


In [136]:
def Weirdness(N_Gram_Google, Lexicon_cat):
    # path_data = '../data/Words_Scrapping/' + str(onlyfiles)
    # Lexicon_cat = pd.read_csv(path_data)
    New_Google = N_Gram_Google[N_Gram_Google[0].isin(Lexicon_cat['Words'])]
    New_Lexicon = Lexicon_cat[Lexicon_cat['Words'].isin(New_Google[0])]
    New_Lexicon = New_Lexicon.sort_values(by=['Words'])
    New_Lexicon['Count'] = New_Lexicon['Count'].astype(float)
    New_Google = New_Google.sort_values(by=[0])
    New_Google[1] = New_Google[1].astype(float)
    New_Google = New_Google.reset_index(drop=True)
    New_Lexicon = New_Lexicon.reset_index(drop=True)
    All_Words = len(N_Gram_Google)
    New_New_Lexicon = New_Lexicon
    New_New_Lexicon['Count'] = New_New_Lexicon['Count'].mul(All_Words)
    New_New_Lexicon['Weirdness'] = (New_Google[1])/New_Lexicon['Count']
    New_New_Lexicon = New_New_Lexicon.sort_values(by=['Weirdness'], ascending=True)
    result = New_New_Lexicon[0:int(len(New_New_Lexicon)*0.2)]
    result = New_Lexicon.reset_index(drop=True)
    return result

In [137]:
N_Gram_Google

,0,1
0,a.165,3.08
1,a.d.a.a.,5.79
2,a.g.s,3.38
3,a.j.u.,2.91
4,a.k.k.,5.33
...,...,...
6156632,zuriickfiihren,4.38
6156633,zuweit,1.87
6156634,zwelff,2.75
6156635,zwierzę,2.61


In [138]:
print(type(N_Gram_Google[0]))

<class 'pandas.core.series.Series'>


In [139]:
New_Google1 = N_Gram_Google[N_Gram_Google[0].isin(df["word_frequency"][0]['Words'])]


In [157]:


# Method 2: Check if value exists (returns boolean)
# print("petra" in df["Lexicon_category_weirdness"][0]["Words"].values)

True


In [140]:
df["Lexicon_category_weirdness"] = df["word_frequency"].apply(lambda x: Weirdness(N_Gram_Google, x))
df["Lexicon_category_weirdness"][0]

,Words,Count,Weirdness
0,angeles,6156637.0,2.128124e-03
1,biography,6156637.0,1.175705e-03
2,born,6156637.0,5.064497e-03
3,breaststroke,18469911.0,7.553366e-06
4,career,6156637.0,4.211039e-03
5,celebrates,6156637.0,1.502947e-04
6,compete,6156637.0,1.258195e-03
7,dutch,6156637.0,2.212812e-03
8,fandom,6156637.0,4.746422e-05
9,feb,6156637.0,1.728005e-04


In [159]:
def calculate_weirdness(word_freq_df, n_gram_google):
    """
    Calculate weirdness score for words using Google N-gram frequencies
    
    Args:
        word_freq_df: DataFrame with word frequencies
        n_gram_google: DataFrame with Google N-gram frequencies
    """
    try:
        # Convert word_freq_df to proper format if it's not already
        if isinstance(word_freq_df, pd.DataFrame) and 'Words' in word_freq_df.columns:
            word_counts = word_freq_df
        else:
            return pd.DataFrame(columns=['Words', 'Count', 'Google_Freq', 'Weirdness'])
            
        # Filter N-gram data to only include words from our frequency list
        new_google = n_gram_google[n_gram_google[0].isin(word_counts['Words'])]
        
        # Filter our frequency list to only include words found in N-gram data
        new_lexicon = word_counts[word_counts['Words'].isin(new_google[0])]
        
        if len(new_lexicon) == 0:
            return pd.DataFrame(columns=['Words', 'Count', 'Google_Freq', 'Weirdness'])
        
        # Sort and reset indices
        new_lexicon = new_lexicon.sort_values(by=['Words']).reset_index(drop=True)
        new_google = new_google.sort_values(by=[0]).reset_index(drop=True)
        
        # Convert frequencies to float
        new_lexicon['Count'] = new_lexicon['Count'].astype(float)
        new_google[1] = new_google[1].astype(float)
        
        # Calculate total words in Google N-gram
        all_words = len(n_gram_google)
        
        # Create result DataFrame
        result = pd.DataFrame({
            'Words': new_lexicon['Words'],
            'Count': new_lexicon['Count'],
            'Google_Freq': new_google[1]
        })
        
        # Calculate weirdness score
        result['Count_Normalized'] = result['Count'] * all_words
        result['Weirdness'] = result['Google_Freq'] / result['Count_Normalized']
        
        # Sort by weirdness
        result = result.sort_values(by='Weirdness', ascending=True)
        
        return result[['Words', 'Count', 'Google_Freq', 'Weirdness']]
        
    except Exception as e:
        print(f"Error in calculate_weirdness: {e}")
        return pd.DataFrame(columns=['Words', 'Count', 'Google_Freq', 'Weirdness'])

# Apply the function to the DataFrame
df["Lexicon_category_weirdness"] = df["word_frequency"].apply(lambda x: calculate_weirdness(x, N_Gram_Google))
df

,query,response,id,urls,snippes_google,scraping_and_procesor,clean_pipeline_web_v,clean_pipeline,chunk_ranking,chunks_filter,...,clean,fix_lista,word_frequency,Lexicon_category_weirdness,response_clean,tokens_response,tokens_in_lexicon,tokens_not_in_lexicon,token_evaluation,lang
0,What did Petra van Staveren win a gold medal for?,Petra van Stoveren won a silver medal in the 2...,val-en-1,[['https://en.wikipedia.org/wiki/Petra_van_Sta...,"[{'title': 'Petra van Staveren - Wikipedia', '...","['Petra van Staveren - Wikipedia', 'Petronella...","[['petra van staveren wikipedia'], ['petronell...",['final women metre breaststroke event summer ...,[{'chuck': 'petra van staveren olympic medals'...,"['petra van staveren olympic medals', 'petra v...",...,"[['petra van staveren wikipedia'], ['petronell...","petra van staveren olympic medals, petra van s...",Words Count 0 petra 1...,Words Count Google_Freq Weird...,"[petra, van, stoveren, win, silver, medal, sum...","{'Petra': 0, 'van': 6, 'Stoveren': 10, 'won': ...","{'Petra': {'position': 0, 'weirdness': 8.09326...","[{'original': 'Stoveren', 'processed': 'stover...","{'important_not_found': [{'token': 'Stoveren',...",EN
1,How many genera does the Erysiphales order con...,The Elysiphale order contains 5 genera.,val-en-2,"[['https://en.wikipedia.org/wiki/Erysiphales',...",[{'title': 'Erysiphales - an overview | Scienc...,['Erysiphales - an overview | ScienceDirect To...,[['erysiphales overview sciencedirect topics']...,['taxonomic manual erysiphales powdery mildews...,[{'chuck': 'date sixteen genera containing spe...,['date sixteen genera containing species descr...,...,[['erysiphales overview sciencedirect topics']...,date sixteen genera containing species describ...,Words Count 0 species ...,Words Count Google_Freq Weir...,"[elysiphale, order, contain, genera, .]","{'The': 0, 'Elysiphale': 4, 'order': 15, 'cont...","{'contains': {'position': 21, 'weirdness': 0.0...","[{'original': 'The', 'processed': None, 'posit...",{'important_not_found': [{'token': 'Elysiphale...,EN


In [160]:
print(df["Lexicon_category_weirdness"][0])

           Words  Count  Google_Freq     Weirdness
25      staveren   11.0        29.81  4.401754e-07
13       grietje    1.0        28.85  4.686000e-06
3   breaststroke    3.0       139.51  7.553366e-06
21         petra   11.0       548.10  8.093261e-06
22    petronella    1.0        59.76  9.706598e-06
29          wiki    1.0       204.09  3.314959e-05
8         fandom    1.0       292.22  4.746422e-05
27       swimmer    2.0       774.64  6.291097e-05
16         medal    3.0      2191.80  1.186687e-04
30     wikipedia    1.0       833.78  1.354278e-04
5     celebrates    1.0       925.31  1.502947e-04
9            feb    1.0      1063.87  1.728005e-04
34         xxiii    1.0      1252.22  2.033935e-04
17        medals    1.0      1317.99  2.140763e-04
28           van   11.0     15852.25  2.340748e-04
20   netherlands    2.0      7772.26  6.312099e-04
11     finishing    1.0      3901.38  6.336869e-04
19        metres    1.0      4348.85  7.063678e-04
18         meter    1.0      63

In [161]:
def tokenize_with_positions(text):
    """
    Tokenize the input string into words and punctuation, tracking the starting position of each token.

    Args:
        text (str): The input string to tokenize.

    Returns:
        dict: A dictionary where keys are tokens and values are their starting positions in the original string.
    """
    import re

    # Define a regex pattern to match words (including contractions) and separate punctuation
    pattern = re.compile(r"\b\w+'?\w*\b|[^\s\w]")

    tokens_with_positions = {}
    for match in pattern.finditer(text):
        token = match.group()
        position = match.start()
        tokens_with_positions[token] = position

    return tokens_with_positions

# # Example Usage
# if __name__ == "__main__":
#     examples = [
#         "Hello, how are you?",
#         "It's a great day!"
#     ]

#     for example in examples:
#         result = tokenize_with_positions(example)
#         print(f"Input: {example}\nOutput: {result}\n")

In [162]:

def clean_response(text):
    preprocessing_methods = {
    # "syntax_patterns_spacy": "special",
    # "syntax_patterns": "special",
    # "token_frequency": "special",
    # "stemming": "text",
    "lemmatization": "text",
    "remove_stopwords": "text",
    # "proper_encoding_ascii": "text",
    #"proper_encoding_utf_8": "text",
    "clean_text": "text"
    }
    data = {
        "text_list": 
            [text]
        ,
        "lang": "en",
        "preprocessing_methods": preprocessing_methods
        }
    
    
    response = requests.post(f"http://localhost:8000/feature/pipeline/", json=data)
    result = response.json()["text"]
    #delete special characters
    
    return result[0].split(" ")
#response1 = requests.post(f"http://localhost:8000/feature/pipeline/", json=data)

def fix_pipeline_list_txt(text):
    list_text = clean_response(text)
    return list_text



df["response_clean"] = df["response"].apply(clean_response)
print(df["response"][0])
print(df["response_clean"][0])


Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.
['petra', 'van', 'stoveren', 'win', 'silver', 'medal', 'summer', 'olympic', 'beijing', 'china', '.']


In [163]:
import spacy
import re
from typing import  List

# Load spacy model once
nlp = spacy.load("en_core_web_md")

def clean_special_characters(text: str) -> str:
    """
    Remove special characters from text while preserving apostrophes and spaces.
    
    Args:
        text (str): Input text
        
    Returns:
        str: Cleaned text
    """
    if not isinstance(text, str):
        return ""
    
    # Remove special characters but keep apostrophes
    cleaned_text = re.sub(r'[^a-zA-Z\'\s]', '', text)
    
    # Remove extra whitespace and convert to lowercase
    cleaned_text = ' '.join(cleaned_text.split()).lower()
    
    return cleaned_text

def preprocess_text(text: str) -> List[str]:
    """
    Preprocess text by cleaning, lemmatizing, and removing stopwords.
    
    Args:
        text (str): Input text
        
    Returns:
        List[str]: List of preprocessed tokens
    """
    try:
        # Clean special characters
        cleaned_text = clean_special_characters(text)
        
        # Process with spaCy
        doc = nlp(cleaned_text)
        
        # Lemmatize and remove stopwords in one pass
        processed_tokens = [
            token.lemma_.lower() 
            for token in doc 
            if not token.is_stop and token.lemma_.strip()
        ]
        
        return processed_tokens
        
    except Exception as e:
        print(f"Error preprocessing text: {str(e)}")
        return []

In [164]:
df["tokens_response"] = df["response"].apply(tokenize_with_positions)
df["tokens_response"]

0    {'Petra': 0, 'van': 6, 'Stoveren': 10, 'won': ...
1    {'The': 0, 'Elysiphale': 4, 'order': 15, 'cont...
Name: tokens_response, dtype: object

In [165]:
import pandas as pd
import re
from typing import List, Dict, Union


def clean_special_characters(text: str) -> str:
    """
    Remove special characters from text while preserving apostrophes and spaces.
    
    Args:
        text (str): Input text
        
    Returns:
        str: Cleaned text
    """
    # Remove special characters but keep apostrophes
    cleaned_text = re.sub(r"[^a-zA-Z'\s]", '', text)
    
    # Remove extra whitespace and convert to lowercase
    cleaned_text = ' '.join(cleaned_text.split()).lower()
    
    return cleaned_text

def preprocess_text(text: str) -> List[str]:
    """
    Preprocess text by cleaning, lemmatizing, and removing stopwords.
    
    Args:
        text (str): Input text
        
    Returns:
        List[str]: List of preprocessed tokens
    """
    try:
        # Clean special characters
        cleaned_text = clean_special_characters(text)
        
        # Process with spaCy
        doc = nlp(cleaned_text)
        
        # Lemmatize and remove stopwords in one pass
        processed_tokens = [
            token.lemma_.lower() 
            for token in doc 
            if not token.is_stop and token.lemma_.strip()
        ]
        
        return processed_tokens
            
    except Exception as e:
        print(f"Error preprocessing text: {str(e)}")
        return []


def categorize_token_spacy(token_text: str) -> Union[str, None]:
    """
    Categorize a token based on its POS tag using spaCy.

    Args:
        token_text (str): The text of the token to categorize.

    Returns:
        str: The category of the token 
             ('quantifier', 'conjunction', 'auxiliary', 'preposition', 
              'number', 'noun', 'pronoun', 'verb', or None)
    """
    doc = nlp(token_text)
    for token in doc:
        pos = token.pos_
    return pos

def map_not_found_tokens_to_positions(tokens_not_found: List[Dict[str, Union[str, None]]], tokens_response: Dict[str, int]) -> List[Dict[str, Union[str, None, List[int]]]]:
    """
    Map not-found tokens to their positions in the original text.
    
    Args:
        tokens_not_found (list): List of dictionaries with 'original' and 'processed' tokens.
        tokens_response (dict): Dictionary mapping tokens to their positions.
        
    Returns:
        list: List of dictionaries with 'original', 'processed', and 'positions'.
    """
    mapped_tokens = []
    
    for token in tokens_not_found:
        original = token.get('original', None)
        processed = token.get('processed', None)
        
        if not original:
            continue  # Skip if original is missing
        
        # Retrieve all positions for the token from tokens_response
        # Handle case-insensitivity by matching lowercased tokens
        positions = [
            pos for tok, pos in tokens_response.items()
            if tok.lower() == original.lower()
        ]
        
        mapped_tokens.append({
            'original': original,
            'processed': processed,
            'positions': positions if positions else []
        })
        
    return mapped_tokens

def evaluate_important_not_found_tokens(mapped_not_found: List[Dict[str, Union[str, List[int]]]]) -> Dict[str, Union[List[Dict[str, Union[str, List[int]]]], Dict[str, float]]]:
    """
    Evaluate not-found tokens to identify important words that were not processed.
    Includes their positions.

    Args:
        mapped_not_found (list): List of dictionaries containing 'original', 'processed', and 'positions' of tokens.

    Returns:
        dict: Dictionary containing important not-found tokens and statistics
    """
    important_not_found = []

    for token in mapped_not_found:
        original = token.get('original', None)
        positions = token.get('positions', [])
        
        if not original:
            continue  # Skip if original is missing
        
        category = categorize_token_spacy(original)
        #types:
        # ADJ: adjective
        # ADP: adposition
        # ADV: adverb
        # AUX: auxiliary
        # CCONJ: coordinating conjunction
        # DET: determiner
        # INTJ: interjection
        # NOUN: noun
        # NUM: numeral
        # PART: particle
        # PRON: pronoun
        # PROPN: proper noun
        # PUNCT: punctuation
        # SCONJ: subordinating conjunction
        # SYM: symbol
        # VERB: verb
        # X: other
        if category not in  {'ADJ', 'ADP', 'ADV', 'AUX ','CCONJ', 'DET', 'PRON', 'PUNCT'}:
            important_not_found.append({
                'token': original,
                'category': category,
                'positions': positions
            })
    return {
        'important_not_found': important_not_found,
    }


def compare_tokens_with_lexicon(tokens_dict: Dict[str, int], lexicon_df: pd.DataFrame) -> Dict[str, Union[Dict[str, Dict[str, Union[int, float, str]]], List[Dict[str, Union[str, int]]]]]:
    """
    Compare tokens to the Lexicon_category_weirdness dataframe and return tokens that are present
    and tokens that are not found, along with their positions.
    
    Args:
        tokens_dict (dict): Dictionary of tokens and their positions.
        lexicon_df (pd.DataFrame): DataFrame containing 'Words' and 'Weirdness'.

    Returns:
        dict: 
            - 'found': Dictionary with tokens as keys and a dictionary of their 'position' and 'weirdness' as values.
            - 'not_found': List of tokens not present in the lexicon, each with positions.
    """
    # Preprocess lexicon words and create a set of the first token from each processed result
    lexicon_df["clean"] = lexicon_df['Words'].apply(lambda x: preprocess_text(x)[0] if preprocess_text(x) else None)
    lexicon_set = set(
        token for token in lexicon_df["clean"] 
        if isinstance(token, str)
    )
    print(lexicon_set)
    # Create a dictionary mapping cleaned words to their weirdness scores
    # Use cleaned words as keys to ensure consistency
    lexicon_weirdness = lexicon_df.set_index('clean')['Weirdness'].to_dict()
    
    found_tokens = {}
    not_found_tokens = []
    
    for token, pos in tokens_dict.items():
        # Preprocess the token
        processed_tokens = preprocess_text(token)
        
        # Use the first processed token
        processed_token = processed_tokens[0] if processed_tokens else None
        
        if processed_token and processed_token.lower() in lexicon_set:
            weirdness_score = lexicon_weirdness.get(processed_token.lower(), None)
            found_tokens[token] = {
                'position': pos,
                'weirdness': weirdness_score,
                'token': token,
            }

        else:
            not_found_tokens.append({
                'original': token,
                'processed': processed_token
            })
    
    # Map not-found tokens to their positions
    not_found_tokens_with_positions = map_not_found_tokens_to_positions(not_found_tokens, tokens_dict)
    
    return {
        'found': found_tokens,
        'not_found': not_found_tokens_with_positions
    }

def apply_token_comparison(row: pd.Series) -> pd.Series:
    """
    Apply the token comparison and mapping for each row in the DataFrame.
    
    Args:
        row (pd.Series): A row from the DataFrame.
    
    Returns:
        pd.Series: Series containing 'tokens_in_lexicon' and 'tokens_not_in_lexicon'.
    """
    if not row["Lexicon_category_weirdness"].empty:
        comparisD= compare_tokens_with_lexicon(
            row["tokens_response"],
            row["Lexicon_category_weirdness"]
        )
        return pd.Series({
            'tokens_in_lexicon': comparison['found'],
            'tokens_not_in_lexicon': comparison['not_found']
        })
    else:
        # Map all tokens as not found with their positions
        tokens_not_found = [
            {'original': token, 'processed': None} 
            for token in row["tokens_response"].keys()
        ]
        tokens_not_found_with_positions = map_not_found_tokens_to_positions(tokens_not_found, row["tokens_response"])
        return pd.Series({
            'tokens_in_lexicon': {},
            'tokens_not_in_lexicon': tokens_not_found_with_positions
        })


# Apply the comparison across the DataFrame
df[["tokens_in_lexicon", "tokens_not_in_lexicon"]] = df.apply(apply_token_comparison, axis=1)

# Apply the evaluation function to the 'tokens_not_in_lexicon' column

# Print evaluation results for the first row
# print_evaluation_results(df['token_evaluation'].iloc[0])
df['token_evaluation'] = df['tokens_not_in_lexicon'].apply(evaluate_important_not_found_tokens)

# Display the updated DataFrame
print("\nUpdated DataFrame:")
print(df[['query', 'response', 'tokens_in_lexicon', 'tokens_not_in_lexicon', 'token_evaluation']].to_string(index=False))

{'swimmer', 'win', 'result', 'petronella', 'petra', 'compete', 'wikipedia', 'medal', 'woman', 'angeles', 'biography', 'grietje', 'staveren', 'metre', 'feb', 'xxiii', 'los', 'summer', 'fandom', 'gold', 'celebrate', 'van', 'career', 'june', 'wiki', 'bear', 'meter', 'final', 'represent', 'breaststroke', 'dutch', 'finish', 'netherlands'}
{'powdery', 'base', 'cook', 'molecular', 'specie', 'erysiphaceae', 'manual', 'family', 'describe', 'taxonomic', 'sixteen', 'ribosomal', 'delimitation', 'comprise', 'genera', 'mildew', 'phylogenetic', 'dna', 'date', 'contain', 'nuclear', 'nov', 'braun', 'fungi', 'relationship', 'erysiphale'}

Updated DataFrame:
                                              query                                                                             response                                                                                                                                                                                                                        

In [169]:
df['token_evaluation'][0]

{'important_not_found': [{'token': 'Stoveren',
   'category': 'NOUN',
   'positions': [10]},
  {'token': 'silver', 'category': 'NOUN', 'positions': [25]},
  {'token': '2008', 'category': 'NUM', 'positions': [45]},
  {'token': 'Olympics', 'category': 'NOUN', 'positions': [57]},
  {'token': 'Beijing', 'category': 'PROPN', 'positions': [69]},
  {'token': 'China', 'category': 'PROPN', 'positions': [78]}]}

## should return 
 'Stoveren': {'position': 10, 'weirdness': None, 'token': 'Stoveren'},
 'China': {'position': 78, 'weirdness': None, 'token': 'China'}}

In [150]:
print(New_Google1[0].str.contains("olympic").any())

False


In [151]:
print("Olympics" in df["Lexicon_category_weirdness"][0]["Words"].values)


False


In [155]:
df['token_evaluation'][1]


{'important_not_found': [{'token': 'Elysiphale',
   'category': 'NOUN',
   'positions': [4]},
  {'token': 'order', 'category': 'NOUN', 'positions': [15]},
  {'token': '5', 'category': 'NUM', 'positions': [30]}]}

In [153]:
import json
import os

# Since all data are in Spanish, assign 'ES' to the 'lang' column
df['lang'] = 'EN'

# ----------------------------
# Step 3: Define Function to Extract Hallucination Spans
# ----------------------------

def extract_hallucination_spans(token_evaluation, model_input):
    """
    Extracts hallucination spans from the token_evaluation dictionary.

    Args:
        token_evaluation (dict): Dictionary containing tokens identified as hallucinations with their positions.

    Returns:
        tuple: A tuple containing two lists:
            - hard_labels: List of [start, end] positions of hallucination spans.
            - soft_labels: List of dictionaries with 'start', 'end', and 'prob' keys.
    """
    hard_labels = []
    soft_labels = []
    
    # Check if 'important_not_found' exists and is a list
    important_not_found = token_evaluation.get('important_not_found', [])
    if not isinstance(important_not_found, list):
        return hard_labels, soft_labels
    
    for token_info in important_not_found:
        token = token_info.get('token', '')
        positions = token_info.get('positions', [])
        
        for start_pos in positions:
            end_pos = start_pos + len(token)
            
            # Append to hard_labels
            hard_labels.append([start_pos, end_pos])
            similarity = sentence_transformers_featurev2(token, model_input)
            
            # Assign a default probability of 0.9 as per user request
            soft_labels.append({
                'start': start_pos,
                'end': end_pos,
                'prob': similarity
            })
    
    return hard_labels, soft_labels

# ----------------------------
# Step 4: Process Each Row and Organize Data
# ----------------------------

# Initialize a list to hold all datapoints
datapoints = []

for idx, row in df.iterrows():
    # Extract necessary fields
    lang = row['lang']
    model_input = row['query']
    model_id = row.get('id', 'unknown_model')
    model_output_text = row['response']
    token_evaluation = row.get('token_evaluation', {})

    
    # Extract hallucination spans
    hard_labels, soft_labels = extract_hallucination_spans(token_evaluation, model_input)
    
    # Create the datapoint dictionary
    datapoint = {
        'lang': lang,
        'model_input': model_input,
        'id': model_id,
        'model_output_text': model_output_text,
        'hard_labels': hard_labels,
        'soft_labels': soft_labels
    }
    
    # Append to the list of datapoints
    datapoints.append(datapoint)

# ----------------------------
# Step 5: Write JSONL File
# ----------------------------

# Define the output directory
output_dir = './output_jsonl/'
os.makedirs(output_dir, exist_ok=True)

# Define the file path for Spanish validation data
file_path = f"{output_dir}pre.jsonl"

# Write the datapoints to the JSONL file
with open(file_path, 'w', encoding='utf-8') as f:
    for datapoint in datapoints:
        json_line = json.dumps(datapoint, ensure_ascii=False)
        f.write(json_line + '\n')

print(f"Written {len(datapoints)} datapoints to {file_path}")

Written 2 datapoints to ./output_jsonl/pre.jsonl


Para evitar trabajar doble es que podemos crear un nuevo dataset que solamente tenga los querys no repetidos, es importante saber que despues se tiene que pasar al dataframe principal, pero con la información repedida para iterar por fila. 